# Multi-seed Training: RoBERTa + Full (E+K) Index

Trains RoBERTa with the Full (E+K) index on IHC, ISHate and Vicomtech for a single seed,
passed via the `TRAINING_SEED` environment variable.

Submit 3 parallel RunAI jobs (seeds 0, 1, 2) alongside the existing seed=42 results
from `weights_rag_best_hp/` to report mean ± std over 4 seeds in the paper.

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm
import warnings
import urllib.request, zipfile, shutil
warnings.filterwarnings('ignore')
from rag import encode, retrieve_top_k_above_threshold

In [2]:
# ── Seed — set via TRAINING_SEED env var at job submission ───────────────────
SEED = int(os.environ.get('TRAINING_SEED', '42'))
print(f'Training seed: {SEED}')

Training seed: 1


In [3]:
# ── Fixed config ─────────────────────────────────────────────────────────────
RAG_DIR         = Path('.')
INDEX_DIR       = RAG_DIR / 'index'
WEIGHTS_DIR     = Path('..') / 'weights_rag_multiseed'
RESULTS_FILE    = Path('..') / 'results' / f'multiseed_results_s{SEED}.json'

MODEL_KEY       = 'roberta'
MODEL_HF_ID     = 'roberta-base'
INDEX_TYPE      = 'full'          # E+K
RETRIEVER_HF_ID = 'sentence-transformers/all-mpnet-base-v2'

# Best HP for RoBERTa from tuning (IHC, full index)
K             = 5
THRESHOLD     = 0.4
MAX_K         = 5
MIN_THRESHOLD = 0.4

MAX_LENGTH    = 256
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3

print(f'Model      : {MODEL_HF_ID}')
print(f'Index      : {INDEX_TYPE}')
print(f'K={K}, threshold={THRESHOLD}')
print(f'Output     : {RESULTS_FILE}')

Model      : roberta-base
Index      : full
K=5, threshold=0.4
Output     : ../results/multiseed_results_s1.json


In [4]:
# ── Seed everything ──────────────────────────────────────────────────────────
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [5]:
# ── Load datasets ────────────────────────────────────────────────────────────
# IHC — data split seed hardcoded at 42 (same test set across all runs)
raw_ihc = load_dataset('tasksource/implicit-hate-stg1', split='train')
splits  = raw_ihc.train_test_split(test_size=0.10, seed=42)

def add_binary_label_ihc(example):
    example['label'] = 0 if example['class'] == 'not_hate' else 1
    return example

train_ihc = splits['train'].map(add_binary_label_ihc)
test_ihc  = splits['test'].map(add_binary_label_ihc)

# ISHate
ishate_raw = load_dataset('BenjaminOcampo/ISHate')

def add_binary_label_ishate(example):
    example['label'] = 0 if example['hateful_layer'] == 'Non-HS' else 1
    return example

train_ishate = ishate_raw['train'].map(add_binary_label_ishate)
test_ishate  = ishate_raw['test'].map(add_binary_label_ishate)

# Vicomtech
_repo_dir      = str(RAG_DIR / 'data' / 'hate-speech-dataset')
_metadata_path = f'{_repo_dir}/annotations_metadata.csv'
_train_dir     = f'{_repo_dir}/sampled_train'
_test_dir      = f'{_repo_dir}/sampled_test'

if not all(os.path.exists(p) for p in [_metadata_path, _train_dir, _test_dir]):
    if os.path.isdir(_repo_dir):
        shutil.rmtree(_repo_dir)
    os.makedirs(str(RAG_DIR / 'data'), exist_ok=True)
    _zip_url  = 'https://github.com/Vicomtech/hate-speech-dataset/archive/refs/heads/master.zip'
    _zip_path = str(RAG_DIR / 'data' / 'hate-speech-dataset.zip')
    urllib.request.urlretrieve(_zip_url, _zip_path)
    with zipfile.ZipFile(_zip_path, 'r') as zf:
        zf.extractall(str(RAG_DIR / 'data'))
    os.rename(str(RAG_DIR / 'data' / 'hate-speech-dataset-master'), _repo_dir)
    os.remove(_zip_path)

_metadata = pd.read_csv(_metadata_path).set_index('file_id')

def _load_vicomtech_split(split_dir):
    rows = []
    for fname in sorted(os.listdir(split_dir)):
        if not fname.endswith('.txt'):
            continue
        file_id = fname[:-4]
        if file_id not in _metadata.index:
            continue
        label_str = _metadata.loc[file_id, 'label']
        if label_str not in ('hate', 'noHate'):
            continue
        with open(os.path.join(split_dir, fname), encoding='utf-8') as f:
            text = f.read().strip()
        rows.append({'text': text, 'label': 1 if label_str == 'hate' else 0})
    return Dataset.from_list(rows)

vicomtech_train = _load_vicomtech_split(_train_dir)
vicomtech_test  = _load_vicomtech_split(_test_dir)

DATASETS = {
    'IHC':       {'train': train_ihc,       'test': test_ihc,       'text_col': 'post'},
    'ISHate':    {'train': train_ishate,     'test': test_ishate,    'text_col': 'text'},
    'Vicomtech': {'train': vicomtech_train,  'test': vicomtech_test, 'text_col': 'text'},
}

print(f'IHC        — train: {len(train_ihc):,}  test: {len(test_ihc):,}')
print(f'ISHate     — train: {len(train_ishate):,}  test: {len(test_ishate):,}')
print(f'Vicomtech  — train: {len(vicomtech_train):,}  test: {len(vicomtech_test):,}')

README.md:   0%|          | 0.00/792 [00:00<?, ?B/s]

implicit_hate_v1_stg1_posts.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21480 [00:00<?, ? examples/s]

Map:   0%|          | 0/19332 [00:00<?, ? examples/s]

Map:   0%|          | 0/2148 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/4367 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4368 [00:00<?, ? examples/s]

Map:   0%|          | 0/55023 [00:00<?, ? examples/s]

Map:   0%|          | 0/4368 [00:00<?, ? examples/s]

IHC        — train: 19,332  test: 2,148
ISHate     — train: 55,023  test: 4,368
Vicomtech  — train: 1,914  test: 478


In [6]:
# ── Self-exclusion lookup ────────────────────────────────────────────────────
chunks_df = pd.read_csv(RAG_DIR / 'chunks' / 'chunks_training.csv')

def strip_label_prefix(text):
    return text.replace('[hate] ', '', 1).replace('[not hate] ', '', 1)

text_to_chunk_id = {
    strip_label_prefix(row.text): int(row.chunk_id)
    for _, row in chunks_df.iterrows()
}
print(f'Self-exclusion lookup: {len(text_to_chunk_id):,} entries')

Self-exclusion lookup: 67,864 entries


In [7]:
# ── Augmentation helpers ─────────────────────────────────────────────────────
def augment_split_cached(hf_dataset, text_col, is_train, ret_model, ret_tokenizer, ret_index, ret_documents):
    """Augment once at MAX_K/MIN_THRESHOLD, storing (text, score) pairs for later filtering."""
    records = []
    for example in tqdm(hf_dataset, desc=f"{'train' if is_train else 'test'}"):
        tweet    = example[text_col]
        chunk_id = text_to_chunk_id.get(tweet) if is_train else None
        neighbors = retrieve_top_k_above_threshold(
            tweet, MIN_THRESHOLD, ret_model, ret_tokenizer, ret_index, ret_documents,
            chunk_id=chunk_id, k=MAX_K, use_mean_pool=True,
        )
        records.append({
            'query':     tweet,
            'neighbors': neighbors,  # list of (text, score)
            'label':     example['label'],
        })
    return records


def filter_records(cached, k, threshold):
    """Filter cached (text, score) neighbors to a specific (k, threshold)."""
    return [
        {
            'query':     r['query'],
            'neighbors': [text for text, score in r['neighbors'] if score >= threshold][:k],
            'label':     r['label'],
        }
        for r in cached
    ]


def tokenize_augmented(records, tokenizer, max_length=MAX_LENGTH):
    sep = tokenizer.sep_token
    texts = [
        f' {sep} '.join([r['query']] + r['neighbors'])
        for r in records
    ]
    labels = [r['label'] for r in records]
    encoded = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=max_length,
    )
    encoded['labels'] = labels
    return Dataset.from_dict(encoded)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
    }

In [8]:
# ── Load retriever and Full index once ───────────────────────────────────────
print(f'Loading retriever: {RETRIEVER_HF_ID} ...')
ret_tokenizer = AutoTokenizer.from_pretrained(RETRIEVER_HF_ID)
ret_model     = AutoModel.from_pretrained(RETRIEVER_HF_ID).eval().to(device)
print(f'Retriever ready on {device}')

index_path    = INDEX_DIR / 'sbert' / f'vdb_{INDEX_TYPE}.faiss'
ret_index     = faiss.read_index(str(index_path))
print(f'FAISS index loaded: {ret_index.ntotal:,} vectors')

with open(INDEX_DIR / f'lookup_{INDEX_TYPE}.json') as f:
    ret_documents = json.load(f)

Loading retriever: sentence-transformers/all-mpnet-base-v2 ...


tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Retriever ready on cuda


FAISS index loaded: 108,816 vectors


In [ ]:
# ── Augment all datasets (cached) ────────────────────────────────────────────
aug_data_cached = {}
for ds_name, ds_cfg in DATASETS.items():
    print(f'\n=== Augmenting {ds_name} ===')
    aug_data_cached[ds_name] = {
        'train': augment_split_cached(ds_cfg['train'], ds_cfg['text_col'], True,
                                      ret_model, ret_tokenizer, ret_index, ret_documents),
        'test':  augment_split_cached(ds_cfg['test'],  ds_cfg['text_col'], False,
                                      ret_model, ret_tokenizer, ret_index, ret_documents),
    }

In [10]:
# ── Train and evaluate on each dataset ───────────────────────────────────────
results = {}

for ds_name in DATASETS:
    set_seed(SEED)  # reset before each dataset for reproducibility
    print(f"\n{'='*60}")
    print(f'Dataset: {ds_name}  |  Model: {MODEL_KEY}  |  Seed: {SEED}')
    print(f"{'='*60}")

    filtered_train = filter_records(aug_data_cached[ds_name]['train'], K, THRESHOLD)
    filtered_test  = filter_records(aug_data_cached[ds_name]['test'],  K, THRESHOLD)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_ID)
    tok_train = tokenize_augmented(filtered_train, tokenizer)
    tok_test  = tokenize_augmented(filtered_test,  tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_HF_ID, num_labels=2)

    save_path = str(WEIGHTS_DIR / MODEL_KEY / 'sbert' / INDEX_TYPE / ds_name / f'seed{SEED}')
    os.makedirs(save_path, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=f'./checkpoints_multiseed/{MODEL_KEY}/{INDEX_TYPE}/{ds_name}/seed{SEED}',
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LEARNING_RATE,
        eval_strategy='epoch',
        save_strategy='no',
        logging_strategy='epoch',
        report_to='none',
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tok_train,
        eval_dataset=tok_test,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f'  Weights saved → {save_path}')

    preds_out = trainer.predict(tok_test)
    preds     = np.argmax(preds_out.predictions, axis=-1)
    labels    = [r['label'] for r in filtered_test]
    print(classification_report(labels, preds, target_names=['Non-HS', 'HS']))

    results[ds_name] = {
        'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
    }

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

del ret_model, ret_tokenizer
if device.type == 'cuda':
    torch.cuda.empty_cache()


Dataset: IHC  |  Model: roberta  |  Seed: 1


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

              precision    recall  f1-score   support

      Non-HS       0.93      0.93      0.93      2681
          HS       0.89      0.88      0.89      1687

    accuracy                           0.91      4368
   macro avg       0.91      0.91      0.91      4368
weighted avg       0.91      0.91      0.91      4368


Dataset: Vicomtech  |  Model: roberta  |  Seed: 1


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1,Macro P,Macro R
1,0.529500,0.561816,0.683719,0.794137,0.707113
2,0.352500,0.417185,0.804238,0.829720,0.807531
3,0.231700,0.356845,0.865770,0.869838,0.866109


  Weights saved → ../weights_rag_multiseed/roberta/sbert/full/Vicomtech/seed1


              precision    recall  f1-score   support

      Non-HS       0.91      0.82      0.86       239
          HS       0.83      0.92      0.87       239

    accuracy                           0.87       478
   macro avg       0.87      0.87      0.87       478
weighted avg       0.87      0.87      0.87       478



In [11]:
# ── Save results to JSON ─────────────────────────────────────────────────────
RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
payload = {
    'seed':    SEED,
    'model':   MODEL_KEY,
    'index':   INDEX_TYPE,
    'results': results,
}
with open(RESULTS_FILE, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Results saved to {RESULTS_FILE}')

Results saved to ../results/multiseed_results_s1.json


In [12]:
# ── Summary table ────────────────────────────────────────────────────────────
rows = []
for ds_name, vals in results.items():
    rows.append({
        'Dataset':   ds_name,
        'F1':        round(vals['macro_f1'], 3),
        'Precision': round(vals['macro_p'],  3),
        'Recall':    round(vals['macro_r'],  3),
    })

df = pd.DataFrame(rows).set_index('Dataset')
print(f'\nRoBERTa (E+K) — seed {SEED}')
display(df)


RoBERTa (E+K) — seed 1


,F1,Precision,Recall
Dataset,,,
IHC,0.804,0.809,0.801
ISHate,0.907,0.908,0.907
Vicomtech,0.866,0.870,0.866
